In [1]:
!pip show accelerate

Name: accelerate
Version: 1.13.0
Summary: Accelerate
Home-page: https://github.com/huggingface/accelerate
Author: The Hugging Face team
Author-email: transformers@huggingface.co
License: Apache
Location: C:\Users\Piyush\AppData\Local\Programs\Python\Python312\Lib\site-packages
Requires: huggingface_hub, numpy, packaging, psutil, pyyaml, safetensors, torch
Required-by: 


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from accelerate import Accelerator

In [3]:
accelerator = Accelerator(mixed_precision="no")

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

dataset = ["Hello world", "Accelerate is powerful"]
inputs = tokenizer(dataset, return_tensors="pt", padding=True)

model, optimizer = accelerator.prepare(model, optimizer)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [5]:
inputs = {k: v.to(accelerator.device) for k, v in inputs.items()}

for step in range(10):
    outputs = model(**inputs, labels=inputs["input_ids"])
    loss = outputs.loss

    accelerator.backward(loss)
    optimizer.step()
    optimizer.zero_grad()

    print(f"step {step} | loss = {loss.item()}")

step 0 | loss = 0.020411724224686623
step 1 | loss = 0.018805954605340958
step 2 | loss = 0.017578192055225372
step 3 | loss = 0.015412140637636185
step 4 | loss = 0.012367243878543377
step 5 | loss = 0.010318463668227196
step 6 | loss = 0.009859904646873474
step 7 | loss = 0.00867187138646841
step 8 | loss = 0.009404020383954048
step 9 | loss = 0.006904354318976402


# Using Datasets

In [6]:
import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from accelerate import Accelerator

In [8]:
accelerator = Accelerator(mixed_precision="no")

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name
)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [9]:
dataset = load_dataset("imdb")
train_data = dataset["train"].shuffle(seed=42).select(range(500))

In [10]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized = train_data.map(tokenize, batched=True)
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [11]:
dataloader = DataLoader(tokenized, batch_size=2, shuffle=True)

In [13]:
model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

In [14]:
model.train()

for step, batch in enumerate(dataloader):
    batch = {k: v.to(accelerator.device) for k, v in batch.items()}

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["input_ids"]
    )

    loss = outputs.loss

    accelerator.backward(loss)
    optimizer.step()
    optimizer.zero_grad()

    if step % 20 == 0:
        print(f"step {step} | loss = {loss.item():.4f}")

    if step == 100:
        break

step 0 | loss = 3.9965
step 20 | loss = 3.7487
step 40 | loss = 3.7976
step 60 | loss = 4.0044
step 80 | loss = 3.8555
step 100 | loss = 3.2929


In [15]:
model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [16]:
prompt = "This movie was"
inputs = tokenizer(prompt, return_tensors="pt").to(accelerator.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

This movie was like this at first, but then I got the chance to watch it on a movie night with my friends. I had no idea how much it cost


## Using accumulation

In [19]:
accelerator = Accelerator(gradient_accumulation_steps=4, mixed_precision="no")

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [20]:
dataset = load_dataset("imdb")
train_data = dataset["train"].shuffle(seed=42).select(range(500))

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized = train_data.map(tokenize, batched=True)
tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

dataloader = DataLoader(tokenized, batch_size=2, shuffle=True)

In [22]:
model, optimizer, dataloader = accelerator.prepare(model, optimizer, dataloader)

In [23]:
model.train()

step = 0

for batch in dataloader:
    batch = {k: v.to(accelerator.device) for k, v in batch.items()}

    with accelerator.accumulate(model):
        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["input_ids"]
        )

        loss = outputs.loss

        accelerator.backward(loss)
        optimizer.step()
        optimizer.zero_grad()

    if step % 20 == 0:
        print(f"step {step} | loss = {loss.item():.4f}")

    step += 1

    if step == 100:
        break

step 0 | loss = 4.6940
step 20 | loss = 2.6595
step 40 | loss = 3.7550
step 60 | loss = 4.2713
step 80 | loss = 3.6976


In [24]:
model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [27]:
prompt = "This movie was"
inputs = tokenizer(prompt, return_tensors="pt").to(accelerator.device)

In [28]:
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

This movie was a little boring. It's not like it was good. The acting was terrible. The plot was so off-track. It was so bad that I
